# Week 3b — A Customer Support Agent

Everything from this week, assembled into one working application: a support agent for **Husky Tech**, a small online electronics store. The agent will look up orders, check return eligibility, answer policy questions, escalate to a human when it should, remember each customer's conversation, and file a structured ticket at the end.

In [ ]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The scenario and the data

A real support agent sits in front of an order database and a policy knowledge base. We mock both with dictionaries so we can focus on the agent. The tool code is the only thing that would change in production; the agent would not.

In [2]:
ORDERS = {
    "HT-1001": {"item": "Wireless headphones", "status": "delivered", "delivered_on": "2026-09-05", "price": 79.99},
    "HT-1002": {"item": "Mechanical keyboard",  "status": "shipped",   "ordered_on": "2026-09-15", "eta": "2026-09-23", "price": 129.00},
    "HT-1003": {"item": "USB-C dock",           "status": "processing","ordered_on": "2026-09-18", "price": 59.50},
}

RETURN_POLICY_DAYS = 30

FAQ = {
    "shipping": "Standard shipping takes 3-5 business days. Orders over $50 ship free.",
    "return":   "Items can be returned within 30 days of delivery for a full refund.",
    "warranty": "All electronics include a one-year manufacturer warranty.",
    "hours":    "Support is available Monday through Friday, 9am to 6pm ET.",
}

## 2. The tools

Four tools, one job each. Remember from Week 2b: the docstring is not a comment; it is how the model decides which tool to call.

In [3]:
from langchain.tools import tool

@tool
def look_up_order(order_id: str) -> str:
    """Look up an order by its id (e.g. HT-1001) and return its current details."""
    order = ORDERS.get(order_id.upper())
    if order is None:
        return f"No order found with id {order_id}. Ask the customer to double-check it."
    return str(order)

In [4]:
from datetime import datetime, date

@tool
def check_return_eligibility(order_id: str) -> str:
    """Check whether an order can still be returned under the 30-day policy. Takes the order id."""
    order = ORDERS.get(order_id.upper())
    if order is None:
        return f"No order found with id {order_id}."
    if order["status"] != "delivered":
        return f"Order {order_id} has not been delivered yet, so the return window has not started."
    delivered = datetime.strptime(order["delivered_on"], "%Y-%m-%d").date()
    days = (date.today() - delivered).days
    if days <= RETURN_POLICY_DAYS:
        return f"Eligible: delivered {days} days ago; returns are accepted within {RETURN_POLICY_DAYS} days of delivery."
    return f"Not eligible: delivered {days} days ago, which is past the {RETURN_POLICY_DAYS}-day window."

In [ ]:
#TODO: write search_faq. It takes the customer's question as a string,
# checks whether any FAQ topic appears in the question (lowercase both sides),
# and returns that topic's answer. If nothing matches, return the list of topics.
# Don't forget the @tool decorator and the docstring.


In [5]:
@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the conversation to a human support agent. Use when the customer is upset, asks for a person, or the available tools cannot resolve the issue. Provide a one-sentence reason."""
    return f"Escalation ticket ESC-1042 created. Reason: {reason}. A human agent will follow up within one business day."

Test the tools directly before handing them to a model, always.

In [6]:
print(look_up_order.invoke({"order_id": "HT-1002"}))
print(check_return_eligibility.invoke({"order_id": "HT-1001"}))
#print(search_faq.invoke({"question": "What are your support hours?"}))
print(escalate_to_human.invoke({"reason": "Customer requested a human agent."}))

{'item': 'Mechanical keyboard', 'status': 'shipped', 'ordered_on': '2026-09-15', 'eta': '2026-09-23', 'price': 129.0}
Eligible: delivered 16 days ago; returns are accepted within 30 days of delivery.
Escalation ticket ESC-1042 created. Reason: Customer requested a human agent.. A human agent will follow up within one business day.


## 3. The system prompt

The tools define what the agent *can* do; the system prompt defines what it *should* do. For a customer-facing agent this is where the guardrails live.

In [7]:
system_prompt = """You are the customer support assistant for Husky Tech, an online electronics store.

Rules:
- Only answer questions about Husky Tech orders, products, and policies. Politely decline anything else.
- Never invent order details. Always use the tools to look up real data.
- If the customer is upset, asks for a person, or the tools cannot resolve the issue, use escalate_to_human.
- Be concise and polite."""

## 4. Assemble the agent

One call: the model, the tools, the guardrails, and a checkpointer so every customer gets their own remembered conversation.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [look_up_order, check_return_eligibility, search_faq, escalate_to_human]

support_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

## 5. Test drives

A fresh `thread_id` for each test keeps them independent.

In [8]:
from langchain.messages import HumanMessage

def ask(text, thread):
    config = {"configurable": {"thread_id": thread}}
    result = support_agent.invoke({"messages": [HumanMessage(content=text)]}, config)
    return result

In [9]:
result = ask("Where is my order HT-1002?", "t1")

for m in result["messages"]:
    m.pretty_print()

NameError: name 'support_agent' is not defined

In [ ]:
result = ask("Can I still return the headphones from order HT-1001?", "t2")
print(result["messages"][-1].text)

In [ ]:
result = ask("What are your support hours?", "t3")
print(result["messages"][-1].text)

In [ ]:
# Guardrail check: off-topic request.
result = ask("Write my history essay for me.", "t4")
print(result["messages"][-1].text)

In [ ]:
# Escalation check.
result = ask("This is the third time I am asking about my missing package and nobody helps. I want to talk to a person.", "t5")

for m in result["messages"]:
    m.pretty_print()

## 6. Memory in action

A natural follow-up refers back with "it". On a **new** thread, the agent cannot know what "it" means; on the **same** thread, the checkpointer makes it work.

In [ ]:
# Wrong thread: no shared history.
result = ask("And when will it arrive?", "t99")
print(result["messages"][-1].text)

In [ ]:
# Same thread as the HT-1002 question from before:
result = ask("And when will it arrive?", "t1")
print(result["messages"][-1].text)

## 7. File the ticket

When the conversation ends, support systems keep a structured record, not a transcript. This is yesterday's `with_structured_output`, pointed at the conversation: transcript in, typed record out.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketRecord(BaseModel):
    """A structured record of a finished support conversation."""
    order_id: str = Field(description="The order discussed, or NONE if no order came up")
    category: Literal["billing", "shipping", "returns", "general", "other"] = Field(
        description="What the conversation was about")
    resolved: bool = Field(description="Whether the customer's question was fully answered")
    summary: str = Field(description="One-sentence summary of the conversation")

In [ ]:
transcript = "\n".join(f"{type(m).__name__}: {m.text}" for m in result["messages"] if m.text)

record = model.with_structured_output(TicketRecord).invoke(
    f"Create a ticket record for this support conversation:\n\n{transcript}")
record

## 8. Chat with your agent

Uncomment and run to talk to the agent live. Type `quit` to stop.

In [ ]:
# config = {"configurable": {"thread_id": "live-demo"}}
# while True:
#     user = input("You: ")
#     if user.lower() in {"quit", "exit"}:
#         break
#     result = support_agent.invoke({"messages": [HumanMessage(content=user)]}, config)
#     print("Agent:", result["messages"][-1].text)

## 9. ICA: add order cancellation

Husky Tech policy: an order can be cancelled only while its status is still `processing`.

1. Write a `cancel_order(order_id)` tool that enforces that rule and updates the order's status to `cancelled`.
2. Add it to the tool list and rebuild the agent.
3. Test on one thread: cancelling HT-1003 should succeed; cancelling HT-1002 should be refused; cancelling HT-1003 a second time should be refused too.
4. Does the system prompt need a new rule? Add one if so.

In [ ]:
@tool
def cancel_order(order_id: str) -> str:
    """TODO: describe what this tool does, its rule, and the argument."""
    # TODO: look the order up; handle the missing-order case
    # TODO: refuse unless status is 'processing'
    # TODO: set the status to 'cancelled' and confirm
    pass


In [ ]:
#TODO: rebuild the agent with the new tool (keep the checkpointer),
# then run the three tests on one thread.


## 10. A chat window

Same trick as W03a.1: `gradio_ui.py` next to this notebook wraps the agent in a real chat interface. Tool calls appear as collapsible entries while the agent works, and the fixed `thread_id` means the conversation is remembered across turns, exactly like one customer's session.

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [ ]:
from gradio_ui import GradioUI

app = GradioUI(support_agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Husky Tech Support")
app.launch()